# Build NJ Daily Features

Build one row per NJ ZIP identifier and date from the real recorded-onset panel. This notebook recalculates the NJ-to-NJ commuting network and generates candidate features. It does not fit models or build future targets.

Inputs in the repository's `Weikun` directory:

- `Real NJ COVID Zip.csv`
- `njny_commute_outputs/njny_commute_zcta_flows_2021.csv`
- `regional_attributes_outputs/njny_zcta_regional_attributes_2021.csv`

Dependencies: pandas and numpy. No encrypted workbook or password is required. No execution output is stored in this delivered notebook.


## Pressure v2 update
Only origins with population > 0 are used, with NJ inbound weights renormalized once. Other local features and the 598-region case panel remain unchanged. The main feature filename is preserved. Existing output is backed up before replacement. Rerun Notebook 10 on this new file; old model outputs describe the earlier pressure definition.

## Definitions and decisions

- `cases_today` equals `onset_record_count`: the selected recorded-onset outcome, not all infections or a newly validated episode count. `unique_person_zip_onset_count` is retained for review and is not an input feature.
- Missing ZIP/date observations are zero under the user's confirmed source rule. Source records lacking onset date remain excluded. Previously saved coverage metadata are preserved; `zero_record_rule_confirmed_by_user=True` records the updated decision separately.
- Features are retrospective and based on final onset-date data. Historical reporting availability is not simulated.
- Population joins currently use exact matching ZIP/ZCTA identifiers. Every row carries `zip_zcta_mapping_status=identifier_match_unvalidated`. A matching string alone does not establish geographic equivalence; regional rates and pressure remain conditional on that assumption.
- NJ-only weights use NJ origins and NJ destinations. Self-links are excluded from pressure, but included in NJ-network employment totals. All selected NJ origins are retained.
- A 7/14-day rate is the cumulative trailing window count divided by population times 100,000. No future or centered windows are used.
- Pressure v2 uses only positive-population NJ origins and renormalizes inbound weights once over those origins. The existing pressure_source_weight_coverage columns now describe the valid-origin network. Original-network coverage and the static original weight retained are saved separately. Initial windows and destinations without valid inflow stay unavailable. No population values or case counts are imputed.


## 1. Functions

Run this cell to define the builder. It does not start the build.

In [ ]:
"""Build retrospective NJ recorded-onset features. No model fitting or future labels."""
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO = Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
DEFAULT_OUTPUT = REPO

def build_features(repo=REPO, output=DEFAULT_OUTPUT):
    repo, output = Path(repo), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    cases = pd.read_csv(repo/'Real NJ COVID Zip.csv', dtype={'zip':'string','state':'string'}, parse_dates=['date'])
    required = {'zip','date','state','onset_record_count'}
    assert required.issubset(cases.columns)
    assert cases.zip.notna().all()
    cases['zip'] = cases.zip.str.strip().str.zfill(5)
    assert cases.zip.str.fullmatch(r'[0-9]{5}').all()
    assert cases.state.eq('NJ').all(), 'Case file must contain NJ only.'
    assert cases.date.notna().all() and cases.date.eq(cases.date.dt.normalize()).all()
    assert not cases.duplicated(['zip','date']).any()
    counts = pd.to_numeric(cases.onset_record_count, errors='raise')
    assert counts.notna().all() and counts.ge(0).all() and counts.eq(np.floor(counts)).all()
    cases['onset_record_count'] = counts.astype('int64')
    cases = cases.sort_values(['zip','date']).reset_index(drop=True)
    ids = sorted(cases.zip.unique())
    dates = pd.date_range(cases.date.min(),cases.date.max())
    assert len(cases)==len(ids)*len(dates)
    assert cases.groupby('zip').size().eq(len(dates)).all()

    attrs = pd.read_csv(repo/'regional_attributes_outputs/njny_zcta_regional_attributes_2021.csv',dtype={'zcta':'string'})
    attrs = attrs.loc[attrs.state.eq('NJ')].copy()
    assert not attrs.zcta.duplicated().any()
    assert set(ids).issubset(set(attrs.zcta)), 'Review unmatched ZIP identifiers before joining.'
    # Matching strings alone do not validate the postal ZIP-to-ZCTA relationship.
    regional = attrs.set_index('zcta').reindex(ids)[['population','land_area_sq_km','population_density_per_sq_km','population_period','area_vintage']].copy()
    for c in ['population','land_area_sq_km','population_density_per_sq_km']:
        regional[c] = pd.to_numeric(regional[c],errors='raise')
    regional['valid_population_denominator'] = regional.population.notna() & regional.population.gt(0)
    regional['zip_zcta_mapping_status'] = 'identifier_match_unvalidated'
    regional['commuting_year'] = 2021
    regional['commuting_job_type'] = 'JT00'

    flows = pd.read_csv(repo/'njny_commute_outputs/njny_commute_zcta_flows_2021.csv',dtype={'origin_zcta':'string','destination_zcta':'string'})
    flows = flows.loc[flows.origin_state.eq('NJ') & flows.destination_state.eq('NJ')].copy()
    assert flows.year.eq(2021).all() and flows.job_type.eq('JT00').all()
    assert set(flows.origin_zcta).issubset(ids) and set(flows.destination_zcta).issubset(ids), 'NJ origins without case rows need explicit review, not silent removal.'
    assert not flows.duplicated(['origin_zcta','destination_zcta']).any()
    assert flows.commute_jobs.notna().all() and flows.commute_jobs.ge(0).all()
    print('Loaded case panel and NJ-to-NJ routes.', flush=True)
    self_link = flows.origin_zcta.eq(flows.destination_zcta)
    routes = flows.loc[~self_link].copy()
    def sums(frame,key):
        return frame.groupby(key).commute_jobs.sum().reindex(ids,fill_value=0)
    regional['resident_jobs_nj_network'] = sums(flows,'origin_zcta').to_numpy()
    regional['workplace_jobs_nj_network'] = sums(flows,'destination_zcta').to_numpy()
    regional['inbound_jobs_external_nj'] = sums(routes,'destination_zcta').to_numpy()
    regional['outbound_jobs_external_nj'] = sums(routes,'origin_zcta').to_numpy()
    regional['internal_jobs'] = sums(flows.loc[self_link],'origin_zcta').to_numpy()
    regional['external_inbound_share_nj_network'] = regional.inbound_jobs_external_nj / regional.workplace_jobs_nj_network.where(regional.workplace_jobs_nj_network.gt(0))
    regional['external_outbound_share_nj_network'] = regional.outbound_jobs_external_nj / regional.resident_jobs_nj_network.where(regional.resident_jobs_nj_network.gt(0))
    assert np.array_equal(regional.resident_jobs_nj_network, regional.outbound_jobs_external_nj+regional.internal_jobs)
    assert np.array_equal(regional.workplace_jobs_nj_network, regional.inbound_jobs_external_nj+regional.internal_jobs)
    routes['inbound_weight_nj'] = routes.commute_jobs / routes.destination_zcta.map(regional.inbound_jobs_external_nj).replace(0,np.nan)
    routes['outbound_weight_nj'] = routes.commute_jobs / routes.origin_zcta.map(regional.outbound_jobs_external_nj).replace(0,np.nan)
    assert routes[['inbound_weight_nj','outbound_weight_nj']].notna().all().all()
    assert np.allclose(routes.groupby('destination_zcta').inbound_weight_nj.sum(),1)
    assert np.allclose(routes.groupby('origin_zcta').outbound_weight_nj.sum(),1)
    routes = routes[['origin_state','origin_zcta','destination_state','destination_zcta','commute_jobs','year','job_type','inbound_weight_nj','outbound_weight_nj']]

    # Renormalize ONCE over valid-population NJ origins; never renormalize by day.
    valid_origin = routes.origin_zcta.map(regional.valid_population_denominator).fillna(False)
    retained = routes.loc[valid_origin].groupby('destination_zcta').inbound_weight_nj.sum().reindex(ids,fill_value=0)
    regional['pressure_original_inbound_weight_retained'] = retained.to_numpy()
    routes['origin_population_valid'] = valid_origin
    routes['inbound_weight_nj_valid_population'] = 0.0
    routes.loc[valid_origin,'inbound_weight_nj_valid_population'] = (
        routes.loc[valid_origin,'inbound_weight_nj'] /
        routes.loc[valid_origin,'destination_zcta'].map(retained)
    )
    assert np.allclose(routes.loc[valid_origin].groupby('destination_zcta').inbound_weight_nj_valid_population.sum(),1)
    panel = cases.copy()
    panel['cases_today'] = panel.onset_record_count
    panel['day_of_week'] = panel.date.dt.dayofweek+1
    panel['month'] = panel.date.dt.month
    panel['onset_day_of_week'] = panel.day_of_week
    panel['onset_day_name'] = panel.date.dt.day_name()
    # Preserve the old flag, and state the user's coverage rule separately.
    panel['zero_record_rule_confirmed_by_user'] = True
    groups = panel.groupby('zip',sort=False)
    for lag in [1,7,14]:
        panel[f'cases_lag_{lag}'] = groups.cases_today.shift(lag)
    for window in [7,14]:
        panel[f'cases_sum_{window}d'] = groups.cases_today.transform(lambda s:s.rolling(window,min_periods=window).sum())
        panel[f'cases_mean_{window}d'] = panel[f'cases_sum_{window}d']/window
        panel[f'case_std_{window}d'] = groups.cases_today.transform(lambda s:s.rolling(window,min_periods=window).std(ddof=1))
    panel['week_over_week_change'] = panel.cases_sum_7d - panel.groupby('zip').cases_sum_7d.shift(7)
    panel['case_velocity'] = groups.cases_today.diff()
    panel['case_acceleration'] = panel.groupby('zip').case_velocity.diff()
    panel['case_jerk'] = panel.groupby('zip').case_acceleration.diff()
    last_positive = panel.date.where(panel.cases_today.gt(0)).groupby(panel.zip).ffill()
    panel['days_since_last_observed_case'] = (panel.date-last_positive).dt.days
    panel = panel.merge(regional.rename_axis('zip').reset_index(),on='zip',how='left',validate='many_to_one')
    denominator = panel.population.where(panel.valid_population_denominator)
    panel['case_rate_today_per_100k'] = panel.cases_today/denominator*100000
    for window in [7,14]:
        panel[f'case_rate_{window}d_per_100k'] = panel[f'cases_sum_{window}d']/denominator*100000

    # W[destination, origin] times origin-rate history. No link-by-day table.
    lookup = {zz:i for i,zz in enumerate(ids)}
    W = np.zeros((len(ids),len(ids)))
    di = routes.destination_zcta.map(lookup).to_numpy()
    oi = routes.origin_zcta.map(lookup).to_numpy()
    W[di,oi] = routes.inbound_weight_nj.to_numpy()
    original_has_inflow = W.sum(axis=1)>0
    assert np.allclose(W.sum(axis=1)[original_has_inflow],1)
    original_W = W.copy()
    W[:,:] = 0
    W[di,oi] = routes.inbound_weight_nj_valid_population.to_numpy()
    has_inflow = W.sum(axis=1)>0
    assert np.allclose(W.sum(axis=1)[has_inflow],1)
    assert np.diag(W).sum()==0
    panel['pressure_definition'] = 'valid_population_nj_origins_renormalized_v2'
    print('Computing pressure using fixed valid-population NJ origin weights.',flush=True)
    for window in [7,14]:
        rate_column = f'case_rate_{window}d_per_100k'
        R = panel.pivot(index='zip',columns='date',values=rate_column).reindex(index=ids,columns=dates).to_numpy(dtype=float)
        usable = np.isfinite(R)
        coverage = W @ usable.astype(float)
        original_coverage = original_W @ usable.astype(float)
        pressure = W @ np.where(usable,R,0)
        complete = np.isclose(coverage,1,rtol=0,atol=1e-10) & has_inflow[:,None]
        pressure[~complete] = np.nan
        coverage[~has_inflow,:] = np.nan
        original_coverage[~original_has_inflow,:] = np.nan
        pressure_col = f'nj_covid_pressure_{window}d'
        coverage_col = f'pressure_source_weight_coverage_nj_{window}d'
        assert panel.zip.tolist()==np.repeat(ids,len(dates)).tolist()
        panel[pressure_col] = pressure.reshape(-1)
        panel[coverage_col] = coverage.reshape(-1)
        panel[f'pressure_original_network_weight_coverage_nj_{window}d'] = original_coverage.reshape(-1)
        chosen = np.argwhere(complete)
        if len(chosen):
            d,t = chosen[len(chosen)//2]
            rr = routes.loc[routes.destination_zcta.eq(ids[d]) & routes.origin_population_valid]
            manual = sum(float(row.inbound_weight_nj_valid_population)*R[lookup[row.origin_zcta],t] for row in rr.itertuples())
            assert np.isclose(manual,pressure[d,t],rtol=1e-10,atol=1e-10)

    assert len(panel)==len(cases) and not panel.duplicated(['zip','date']).any()
    assert panel.cases_today.sum()==cases.onset_record_count.sum()
    assert panel.groupby('zip').cases_sum_7d.apply(lambda s:s.iloc[:6].isna().all()).all()
    assert panel.groupby('zip').cases_lag_14.apply(lambda s:s.iloc[:14].isna().all()).all()
    numeric = panel.select_dtypes(include='number')
    assert not np.isinf(numeric.to_numpy(dtype=float)).any()
    missing = panel.isna().sum().rename('missing_rows').to_frame()
    missing['missing_share'] = missing.missing_rows/len(panel)
    missing.rename_axis('column').to_csv(output/'feature_missingness.csv')
    regional.rename_axis('zip').to_csv(output/'nj_regional_features_2021.csv')
    routes.to_csv(output/'nj_commuting_weights_2021.csv',index=False)
    panel.head(30).to_csv(output/'nj_daily_features_preview.csv',index=False)
    print('Saving compressed feature panel.',flush=True)
    target = output/'nj_daily_features.csv.gz'
    if target.exists():
        from datetime import datetime, timezone
        import shutil
        backup = output/'feature_backups'
        backup.mkdir(exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        shutil.copy2(target,backup/f'nj_daily_features_before_rebuild_{stamp}.csv.gz')
    temporary = output/'nj_daily_features.building.csv.gz'
    panel.to_csv(temporary,index=False,compression='gzip')
    temporary.replace(target)
    summary = {
        'rows':len(panel),'columns':len(panel.columns),'regions':len(ids),'days':len(dates),
        'start':str(dates.min().date()),'end':str(dates.max().date()),
        'case_count_sum':int(panel.cases_today.sum()),'nj_routes_including_self':len(flows),'nj_external_routes':len(routes),
        'nj_network_jobs':int(flows.commute_jobs.sum()),'regions_without_external_inflow':int((~original_has_inflow).sum()),
        'regions_without_valid_origin_inflow':int((~has_inflow).sum()),
        'regions_with_valid_origin_pressure':int(has_inflow.sum()),
        'regions_without_valid_population':int((~regional.valid_population_denominator).sum()),
        'pressure_nonmissing_rows':{str(w):int(panel[f'nj_covid_pressure_{w}d'].notna().sum()) for w in [7,14]},
        'geographic_mapping':'Exact ZIP/ZCTA identifier match only; correspondence remains unvalidated.',
        'target_basis':'Retrospective recorded illness-onset rows after exact-row deduplication, not all infections.',
        'zero_rule':'Absent source ZIP/date groups are zero as confirmed by user; missing onset rows remain excluded.',
        'source_coverage_flag':'Original surveillance_coverage_confirmed column retained unchanged as prior metadata; user zero-rule confirmation is a separate field.',
        'network_scope':'NJ-to-NJ only; external excludes self-links; shares use NJ-network denominators.',
        'pressure_policy':'v2: fixed NJ valid-population origins only, inbound weights renormalized over valid origins. Coverage column uses this new denominator; original-network coverage and static retained-weight fraction are preserved separately. No valid inflow is unavailable.',
        'status':'Candidate features computed. No future labels, model fitting or final feature selection. Mapping and infection-episode definitions still require review.',
    }
    (output/'build_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(json.dumps(summary,indent=2),flush=True)
    return panel, routes, summary



## 2. Set paths and run

Update `PROJECT` if needed. If this notebook is opened in the local `Weikun` folder, `Path.cwd()` should identify that folder. Outputs are saved directly in PROJECT (Weikun). The main filename stays nj_daily_features.csv.gz and is replaced after a successful write. Before replacement, the existing main file is copied into feature_backups. Building and exporting may take several minutes.


In [ ]:
PROJECT = Path.cwd()
assert (PROJECT / "Real NJ COVID Zip.csv").is_file(), "Set PROJECT to your repository's Weikun directory."
OUTPUT = PROJECT

panel, nj_routes, summary = build_features(PROJECT, OUTPUT)


## 3. Inspect results

Inspect the summary and missingness before selecting model inputs. Initial lag/window features are intentionally missing. Rolling means and totals are redundant rescalings; their simultaneous availability is for candidate selection, not an instruction to use all columns. Fixed region effects absorb static covariates in a fixed-effects linear model.

IDs, dates, diagnostic counts, source flags and mapping/coverage metadata are not automatically predictive inputs. `target_weekday` and `target_month` will be created when future horizons are defined. No future targets are included in this feature build.


In [ ]:
display(pd.Series(summary, name="value").to_frame())
display(panel[[
    "zip", "date", "cases_today", "cases_lag_1", "cases_sum_7d",
    "case_rate_7d_per_100k", "nj_covid_pressure_7d",
    "pressure_source_weight_coverage_nj_7d"
]].head(25))
display(pd.read_csv(OUTPUT / "feature_missingness.csv"))
print("Saved to:", OUTPUT.resolve())


## 4. Read the exported feature panel later

The main file is a compressed CSV. Pandas reads it directly without manual extraction. It contains the same region-day rows; compression only reduces storage. Keep ZIP identifiers as strings.


In [ ]:
# Run later when you need to reload the saved panel.
# panel = pd.read_csv(
#     OUTPUT / "nj_daily_features.csv.gz",
#     dtype={"zip": "string"},
#     parse_dates=["date"],
# )


## Next steps

Review geographic correspondence and the original inbound weight retained after excluding zero-population sources. This is a changed pressure definition; rerun both matched models before comparing results. Resolve the target counting unit, choose the initial NJ destinations, and build labels for days 1–14. Use common chronological train/validation/test cutoffs and remove label windows crossing those boundaries. Choose features using training and validation only. Compare models with and without pressure on the same eligible region-days.

Outputs are saved in the specified local GitHub folder; no commit or push is performed. Before committing this notebook, clear execution outputs containing real-data previews. This notebook does not upload or publish any data.
